# Projeto Final — Visão Computacional
## Tema 3: Lesões de pele com incerteza (HAM10000 / ISIC 2018 Task 3)
**Aluna:** Alice De Muylder Oliveira

**Pergunta:** uma CNN treinada no HAM10000 consegue classificar as 7 categorias diagnósticas com macro-F1 competitivo e bom recall de melanoma, sinalizando quando está incerta e mostrando (Grad-CAM) em que região baseou a decisão?

Este notebook executa o fluxo completo chamando os módulos em `src/` e os scripts `train.py`, `compare.py`, `evaluate.py` e `predict.py` (que também rodam sozinhos pelo terminal).

> Pré-requisito: a pasta `outputs_M2/` (train/validation/test.csv do M2) e as imagens em `data/` — ver README.

In [ ]:
import sys, json, torch, pandas as pd
from pathlib import Path
from IPython.display import Image as Show, display
sys.path.insert(0, str(Path.cwd()))
import config as C
print('torch', torch.__version__, '| GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NÃO (CPU)')

# True = teste rápido (poucas imagens, 1 época) só para checar se tudo roda
QUICK_TEST = False
OVR = dict(epochs=1) if QUICK_TEST else {}
LIMIT = 140 if QUICK_TEST else None

## 1. Problema e dados
Manifestos do M2: split 70/15/15 por `capture_group` (lesion_id + quase-duplicatas), seed 42, 2 duplicatas exatas excluídas. A checagem de vazamento é refeita aqui.

In [ ]:
from src.data import load_split, check_no_leakage
dfs = {s: load_split(s, LIMIT) for s in ['train', 'validation', 'test']}
check_no_leakage(dfs); print('Sem vazamento entre splits ✔')
dist = pd.DataFrame({s: d.label.value_counts() for s, d in dfs.items()}).loc[C.CLASSES]
dist['total'] = dist.sum(1); dist

### Pré-processamento e aumento (política do M2 — aumentos só no treino)

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image
from src.data import build_transforms, denormalize
p = dfs['train'].path.iloc[0]; tr = build_transforms(True); ev = build_transforms(False)
fig, ax = plt.subplots(1, 6, figsize=(16, 3))
ax[0].imshow(Image.open(p)); ax[0].set_title('original 600×450')
ax[1].imshow(denormalize(ev(Image.open(p).convert('RGB'))).permute(1,2,0)); ax[1].set_title('val/teste (224)')
for k in range(2, 6):
    ax[k].imshow(denormalize(tr(Image.open(p).convert('RGB'))).permute(1,2,0)); ax[k].set_title('treino (aumento)')
for a in ax: a.axis('off')
plt.show()

## 2. Baseline neural e 3. Experimentos
| Run | O que muda | Hipótese |
|---|---|---|
| **B0_baseline** | ResNet-18 ImageNet, CE, AdamW lr 1e-4, wd 1e-4, batch 32, até 15 épocas, cosine, dropout 0.2, early stopping (paciência 5) | referência |
| **E1_perda_ponderada** | CE ponderada por classe (pesos do treino) | aumenta macro-F1 e recall de mel/df/vasc, com algum custo em `nv` |
| **E2_do_zero** | sem pesos ImageNet | com ~7 mil imagens, transferência de aprendizado deve ser claramente melhor |
| E3 (opcional) | EfficientNet-B0 + perda ponderada | arquitetura mais eficiente melhora a macro-F1 |

O melhor checkpoint de cada run é escolhido pela **macro-F1 de validação**. O teste não é usado aqui.

In [ ]:
from src.engine import train_experiment
for exp in ['B0_baseline', 'E1_perda_ponderada', 'E2_do_zero']:   # adicione 'E3_efficientnet_ponderada' se der tempo
    train_experiment(exp, OVR, LIMIT)

### Comparação (somente validação) e escolha do modelo final

In [ ]:
import compare; compare.main()
display(pd.read_csv(C.RESULTS_DIR / 'comparacao_validacao.csv').round(3))
display(Show(C.RESULTS_DIR / 'figuras/fig_curvas_treino_validacao.png'))
display(Show(C.RESULTS_DIR / 'figuras/fig_comparacao_metricas.png'))

## 4. Resultados no teste e análise de erros
O conjunto de teste é usado **uma única vez**, para o modelo final. O limiar de incerteza é definido na validação (20% menos confiantes = encaminhar).

In [ ]:
import evaluate, sys
sys.argv = ['evaluate.py'] + (['--limit', str(LIMIT)] if LIMIT else [])
evaluate.main()
for f in ['fig_matriz_confusao_teste', 'fig_metricas_por_classe_teste', 'fig_incerteza_cobertura',
          'fig_acertos_gradcam', 'fig_erros_gradcam']:
    fp = C.RESULTS_DIR / 'figuras' / f'{f}.png'
    if fp.exists(): display(Show(fp))

## 5. Demonstração: imagem → pré-processamento → modelo → resultado
Imagens do teste (nunca vistas no treino). Para uma imagem externa: `pred('caminho.jpg')`.

In [ ]:
from predict import SkinLesionPredictor
pred = SkinLesionPredictor()
C.DEMO_DIR.mkdir(exist_ok=True)
for _, r in dfs['test'].sample(4, random_state=7).iterrows():
    out = pred(r.path); f = C.DEMO_DIR / f'demo_{r.sample_id}.png'
    pred.save_figure(out, f, r.label); display(Show(f))

## Conclusão
*(preencher com os números de `results/`)*
- Resposta à pergunta: ...
- Resultados satisfatórios? ...
- O que os experimentos ensinaram: ...
- Limitações: desbalanceamento (df com 7 imagens no teste), sub-representação de peles escuras, possível atalho de brilho/nitidez (M2), segmentação não treinada (dados de Task 1 e Task 3 disjuntos). Uso educacional — não substitui avaliação clínica.